# Time Series Analysis by Primary Diagnosis (AutoML)

Time series by primary diagnosis: selection of viable series by temporal density, **AutoML** (FLAML) modeling to choose the best model per series, and export of summaries, charts, and evaluation metrics.

## 1. Import data

**Summary:** Loads the classified Excel, filters to the period September 2023–December 2025, and aggregates counts by **week** and **primary_diagnosis** (disease). Output is a weekly table (week, doenca, casos) used in the next steps.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().resolve()
for _ in range(6):
    if (ROOT / "data").exists() and (ROOT / "notebooks").exists():
        break
    ROOT = ROOT.parent if ROOT.parent != ROOT else ROOT

PATH_INPUT = ROOT / "data" / "results" / "agent_outputs" / "agent_outputs_dataset_sintomas_grupos_classificado.xlsx"
PATH_OUTPUT = ROOT / "data" / "results" / "time_series_outputs"
PATH_SUMMARIES = PATH_OUTPUT / "summaries"
PATH_CHARTS = PATH_OUTPUT / "charts"
os.makedirs(PATH_OUTPUT, exist_ok=True)
os.makedirs(PATH_SUMMARIES, exist_ok=True)
os.makedirs(PATH_CHARTS, exist_ok=True)

DATE_START = "2023-09-01"
DATE_END = "2025-12-31"
FREQ = "W-MON"
TRAIN_RATIO = 0.70

df = pd.read_excel(PATH_INPUT, engine="openpyxl")
df["report_created_at"] = pd.to_datetime(df["report_created_at"], errors="coerce")
df = df.dropna(subset=["report_created_at", "primary_diagnosis"])
df = df[(df["report_created_at"] >= DATE_START) & (df["report_created_at"] <= DATE_END)]
df["week"] = df["report_created_at"].dt.to_period(FREQ).dt.start_time

if "user_id" in df.columns:
    weekly = df.groupby(["week", "primary_diagnosis"])["user_id"].nunique().reset_index()
else:
    weekly = df.groupby(["week", "primary_diagnosis"]).size().reset_index(name="n")
    weekly["user_id"] = weekly["n"]
weekly.columns = ["week", "doenca", "casos"]

print(f"Registros semanais: {len(weekly)}")
print(f"Doenças: {weekly["doenca"].nunique()}")
weekly.head(10)

## 2. Select series with viable density

**Summary:** For each disease, computes **temporal density** (share of weeks with at least one case) and total cases. Keeps only series above minimum density and minimum total cases so that time series modeling is feasible.

In [ ]:
weeks_full = pd.date_range(start=DATE_START, end=DATE_END, freq=FREQ)
n_weeks = len(weeks_full)

density_list = []
for doenca, grp in weekly.groupby("doenca"):
    n_weeks_with_data = grp["week"].nunique()
    total_casos = grp["casos"].sum()
    density = n_weeks_with_data / n_weeks if n_weeks else 0
    density_list.append({"doenca": doenca, "density": density, "total_casos": total_casos, "n_weeks_with_data": n_weeks_with_data})

density_df = pd.DataFrame(density_list)
MIN_DENSITY = 0.10
MIN_CASOS = 20
viable = density_df[(density_df["density"] >= MIN_DENSITY) & (density_df["total_casos"] >= MIN_CASOS)]["doenca"].tolist()
print(f"Séries viáveis (densidade >= {MIN_DENSITY}, total_casos >= {MIN_CASOS}): {len(viable)}")
density_df.sort_values("density", ascending=False).head(15)

## 3. AutoML modeling (70% train / 30% test)

**Summary:** For each viable series, applies a chronological 70/30 split, fits **FLAML** (ts_forecast) to select the best model automatically, and generates test-period forecasts and metrics (RMSE, MAE, MAPE). Detects poor-quality series (e.g. high MAPE or 95% CI crossing zero) and stores results for summaries, charts, and the metrics file.

In [ ]:
from flaml import AutoML
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

def safe_name(s, max_len=50):
    return s.replace("/", "-").replace("\\", "-")[:max_len]

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

automl_results = []
TIME_BUDGET = 90

for i, doenca in enumerate(viable):
    print(f"  [{i+1}/{len(viable)}] {doenca}")
    sub = weekly[weekly["doenca"] == doenca].set_index("week")["casos"]
    series = sub.reindex(pd.DatetimeIndex(weeks_full)).fillna(0)
    if series.sum() > 0 and series.eq(0).any():
        series = series.replace(0, np.nan).interpolate(method="linear").fillna(0)
    series = series.astype(float)
    n = len(series)
    cut = int(n * TRAIN_RATIO)
    if cut < 2 or n - cut < 1:
        continue
    train_series = series.iloc[:cut]
    test_series = series.iloc[cut:]
    train_df = train_series.reset_index()
    train_df.columns = ["timestamp", "value"]
    period = len(test_series)

    try:
        automl = AutoML()
        automl.fit(
            dataframe=train_df,
            label="value",
            period=period,
            task="ts_forecast",
            time_budget=TIME_BUDGET,
            metric="mape",
            eval_method="holdout",
            verbose=0,
        )
        X_test = test_series.index.to_frame(index=False)
        X_test.columns = ["timestamp"]
        y_pred = automl.predict(X_test)
        y_pred = np.maximum(np.asarray(y_pred).flatten(), 0)
        y_test = test_series.values
    except Exception as e:
        print(f"    Erro: {e}")
        continue

    res_std = np.nanstd(y_test - y_pred)
    if np.isnan(res_std) or res_std <= 0:
        res_std = np.nanstd(y_test) or 1.0
    lower = np.maximum(y_pred - 1.96 * res_std, 0)
    upper = y_pred + 1.96 * res_std

    best_estimator = getattr(automl, "model", None)
    try:
        cfg = getattr(automl, "best_config", None)
        model_name = cfg.get("learner") if isinstance(cfg, dict) else None
    except Exception:
        model_name = None
    if not model_name:
        est = getattr(best_estimator, "estimator", best_estimator)
        model_name = type(est).__name__ if est is not None else "AutoML"
    model_name = str(model_name)

    summary_text = ""
    try:
        est = getattr(best_estimator, "estimator", best_estimator)
        if hasattr(est, "summary"):
            s = est.summary()
            summary_text = s.as_text() if hasattr(s, "as_text") else str(s)
        else:
            summary_text = f"Model: {model_name}"
    except Exception:
        summary_text = f"Model: {model_name}"

    rmse_val = rmse(y_test, y_pred)
    mape_val = mape(y_test, y_pred)
    mae_val = mean_absolute_error(y_test, y_pred)
    ci_crosses_zero = np.any((y_pred - 1.96 * res_std) <= 0)
    bad_metrics = (np.isfinite(mape_val) and mape_val > 100) or (np.isfinite(rmse_val) and rmse_val > 2 * np.nanmean(y_test))
    if ci_crosses_zero or bad_metrics:
        obs = "Série muito ruim: "
        parts = []
        if bad_metrics:
            parts.append("métricas de desempenho elevadas (MAPE>100% ou RMSE muito alto)")
        if ci_crosses_zero:
            parts.append("intervalo de confiança 95% passando pelo zero")
        observacoes = obs + "; ".join(parts) + "."
    else:
        observacoes = ""

    automl_results.append({
        "doenca": doenca,
        "modelo": model_name,
        "y_test": y_test,
        "y_pred": y_pred,
        "lower_95": lower,
        "upper_95": upper,
        "test_index": test_series.index,
        "summary_text": summary_text,
        "RMSE": rmse_val,
        "MAE": mae_val,
        "MAPE": mape_val,
        "observações": observacoes,
    })

print(f"Modeladas com sucesso: {len(automl_results)} séries.")

## 4. Save model summaries

**Summary:** Writes one Excel file per series with the chosen model’s summary (e.g. statsmodels output or model name). Path: `data/results/time_series_outputs/summaries/{doenca}_{modelo}_summary.xlsx`.

In [ ]:
for r in automl_results:
    doenca = r["doenca"]
    modelo = r["modelo"]
    summary_text = r["summary_text"]
    fname = f"{safe_name(doenca)}_{safe_name(modelo)}_summary.xlsx"
    path = PATH_SUMMARIES / fname
    pd.DataFrame({"summary": [summary_text]}).to_excel(path, index=False, engine="openpyxl")
print(f"Salvos {len(automl_results)} summaries em {PATH_SUMMARIES}")

## 5. Save charts (test series + forecast + 95% CI)

**Summary:** For each series, saves one chart showing only the **test period**: observed values, point forecast, and 95% confidence band. Path: `data/results/time_series_outputs/charts/{doenca}_{modelo}_chart.png`.

In [ ]:
for r in automl_results:
    doenca = r["doenca"]
    modelo = r["modelo"]
    idx = r["test_index"]
    x_ax = idx.to_timestamp() if hasattr(idx, "to_timestamp") else idx
    y_test = r["y_test"]
    y_pred = r["y_pred"]
    lower = r["lower_95"]
    upper = r["upper_95"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x_ax, y_test, label="Teste (observado)", color="C0")
    ax.plot(x_ax, y_pred, label="Forecast", color="C1")
    ax.fill_between(x_ax, lower, upper, alpha=0.3, color="C1")
    ax.set_title(f"{doenca} — {modelo}")
    ax.legend(loc="best", fontsize=8)
    ax.set_ylabel("Casos")
    plt.tight_layout()
    fname = f"{safe_name(doenca)}_{safe_name(modelo)}_chart.png"
    fig.savefig(PATH_CHARTS / fname, dpi=120, bbox_inches="tight")
    plt.close(fig)
print(f"Salvos {len(automl_results)} gráficos em {PATH_CHARTS}")

## 6. Save evaluation metrics

**Summary:** Exports a single Excel file with one row per series: disease, chosen model, RMSE, MAE, MAPE, and observations (e.g. notes when the series is poor quality). Path: `data/results/time_series_outputs/evaluation_metrics_all_models.xlsx`.

In [ ]:
metrics_df = pd.DataFrame([
    {"doenca": r["doenca"], "modelo_escolhido": r["modelo"], "RMSE": r["RMSE"], "MAE": r["MAE"], "MAPE": r["MAPE"], "observações": r.get("observações", "")}
    for r in automl_results
])
path_metrics = PATH_OUTPUT / "evaluation_metrics_all_models.xlsx"
metrics_df.to_excel(path_metrics, index=False, engine="openpyxl")
print(f"Salvo: {path_metrics}")
metrics_df.head(20)